<a href="https://colab.research.google.com/github/NU8B/Vbot/blob/main/colab/StyleTTS_ft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install packages and download models

In [1]:
%%shell
git clone https://github.com/yl4579/StyleTTS2.git
cd StyleTTS2
pip install datasets SoundFile torchaudio munch torch pydub pyyaml librosa nltk matplotlib accelerate transformers phonemizer einops einops-exts tqdm typing-extensions git+https://github.com/resemble-ai/monotonic_align.git
sudo apt-get install espeak-ng
git-lfs clone https://huggingface.co/yl4579/StyleTTS2-LibriTTS
mv StyleTTS2-LibriTTS/Models .

Cloning into 'StyleTTS2'...
remote: Enumerating objects: 372, done.
remote: Total 372 (delta 0), reused 0 (delta 0), pack-reused 372 (from 1)
Receiving objects: 100% (372/372), 133.98 MiB | 15.51 MiB/s, done.
Resolving deltas: 100% (199/199), done.
Updating files: 100% (48/48), done.
  Cloning https://github.com/resemble-ai/monotonic_align.git to /tmp/pip-req-build-v3zk2thi
  Running command git clone --filter=blob:none --quiet https://github.com/resemble-ai/monotonic_align.git /tmp/pip-req-build-v3zk2thi
  Resolved https://github.com/resemble-ai/monotonic_align.git to commit 78b985be210a03d08bc3acc01c4df0442105366f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 16.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Download dataset



In [2]:
%cd StyleTTS2
!rm -rf Data

!gdown --id 1t5h-rhIeUMjfdwUPx-TMIpZtysiMVVgV
!unzip Data.zip

/content/StyleTTS2
/usr/local/lib/python3.10/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1t5h-rhIeUMjfdwUPx-TMIpZtysiMVVgV
From (redirected): https://drive.google.com/uc?id=1t5h-rhIeUMjfdwUPx-TMIpZtysiMVVgV&confirm=t&uuid=66d85b8b-e058-4fd4-917f-6cc7f8467265
To: /content/StyleTTS2/Data.zip
100% 356M/356M [00:05<00:00, 63.0MB/s]
Archive:  Data.zip
  inflating: Data/OOD_texts.txt      
  inflating: Data/train_list.txt     
  inflating: Data/val_list.txt       
   creating: Data/wavs/
  inflating: Data/wavs/0003.wav      
  inflating: Data/wavs/0004.wav      
  inflating: Data/wavs/0005.wav      
  inflating: Data/wavs/0006.wav      
  inflating: Data/wavs/0007.wav      
  inflating: Data/wavs/0008.wav      
  inflating: Data/wavs/0009.wav      
  inflating: Data/wavs/0010.wav  

### Change the finetuning config

Depending on the GPU you got, you may want to change the bacth size, max audio length, epiochs and so on.

In [3]:
config_path = "Configs/config_ft.yml"

import yaml

config = yaml.safe_load(open(config_path))

In [11]:
config["data_params"]["root_path"] = "Data/wavs"
config["batch_size"] = 2  # not enough RAM
config["max_len"] = 120
config["epochs"] = 5
config["loss_params"][
    "joint_epoch"
] = 110  # we do not do SLM adversarial training due to not enough RAM

with open(config_path, "w") as outfile:
    yaml.dump(config, outfile, default_flow_style=True)

### Start finetuning


In [12]:
!accelerate launch --mixed_precision=fp16 --num_processes=1 train_finetune_accelerate.py --config_path ./Configs/config_ft.yml | tqdm --total 100 --desc "Training Progress"

Training Progress:   0% 0/100 [00:00<?, ?it/s]The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2024-12-21 12:11:51.950827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-21 12:11:51.969507: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-21 12:11:51.974903: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-21 12:11:51.988458: I t

In [13]:
from huggingface_hub import login, HfApi, create_repo
import os
import json
import torch
from datetime import datetime
import shutil
import glob
import yaml
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def convert_tensor_to_list(obj):
    """Convert torch tensors to lists recursively"""
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    elif isinstance(obj, dict):
        return {key: convert_tensor_to_list(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensor_to_list(item) for item in obj]
    elif isinstance(obj, tuple):
        return tuple(convert_tensor_to_list(item) for item in obj)
    else:
        return obj


# Plot training metrics
def plot_training_metrics(log_file, save_dir):
    metrics = {
        "train_loss": [],
        "val_loss": [],
        "dur_loss": [],
        "F0_loss": [],
        "epochs": [],
    }

    try:
        with open(log_file, "r") as f:
            lines = f.readlines()

        for line in lines:
            if "Validation loss" in line:
                # Extract metrics from validation lines
                parts = line.split(",")
                val_loss = float(parts[0].split(":")[-1])
                dur_loss = float(parts[1].split(":")[-1])
                f0_loss = float(parts[2].split(":")[-1])

                metrics["val_loss"].append(val_loss)
                metrics["dur_loss"].append(dur_loss)
                metrics["F0_loss"].append(f0_loss)
                metrics["epochs"].append(len(metrics["val_loss"]))

        # Create plots
        plt.figure(figsize=(15, 5))

        plt.subplot(131)
        plt.plot(metrics["epochs"], metrics["val_loss"])
        plt.title("Validation Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        plt.subplot(132)
        plt.plot(metrics["epochs"], metrics["dur_loss"])
        plt.title("Duration Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        plt.subplot(133)
        plt.plot(metrics["epochs"], metrics["F0_loss"])
        plt.title("F0 Loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")

        plt.tight_layout()
        plt.savefig(os.path.join(save_dir, "training_metrics.png"))
        plt.close()

        return metrics

    except Exception as e:
        print(f"Error plotting metrics: {e}")
        return None


# Login to Hugging Face and setup repository
from google.colab import userdata
try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
repo_name = "nonoJDWAOIDAWKDA/new2_ft_StyleTTS2"
login(token) if token else login()
api = HfApi()

try:
    create_repo(repo_name, exist_ok=True, token=token, repo_type="model")
except Exception as e:
    print(f"Repository already exists or error creating: {e}")

# Load config and checkpoint
config_path = "Configs/config_ft.yml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
config = convert_tensor_to_list(config)

checkpoint_dir = "Models/LJSpeech"
files = [f for f in os.listdir(checkpoint_dir) if f.endswith(".pth")]
if not files:
    raise ValueError(f"No checkpoint files found in {checkpoint_dir}")
latest_checkpoint = sorted(files, key=lambda x: int(x.split("_")[-1].split(".")[0]))[-1]
checkpoint_path = os.path.join(checkpoint_dir, latest_checkpoint)
print(f"Loading checkpoint: {checkpoint_path}")

checkpoint = torch.load(checkpoint_path, map_location="cpu")

# Prepare files for upload
temp_dir = "temp_model"
if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir)

# Create directory structure
os.makedirs(os.path.join(temp_dir, "Utils/ASR"), exist_ok=True)
os.makedirs(os.path.join(temp_dir, "Utils/JDC"), exist_ok=True)
os.makedirs(os.path.join(temp_dir, "Utils/PLBERT"), exist_ok=True)

# Create .gitattributes first
with open(os.path.join(temp_dir, ".gitattributes"), "w") as f:
    f.write("*.pth filter=lfs diff=lfs merge=lfs -text\n")
    f.write("*.t7 filter=lfs diff=lfs merge=lfs -text\n")

# Save model components
print("Saving model components...")
required_components = [
    "bert",
    "bert_encoder",
    "decoder",
    "diffusion",
    "mpd",
    "msd",
    "predictor",
    "predictor_encoder",
    "style_encoder",
    "text_aligner",
    "text_encoder",
    "pitch_extractor",
    "wd",
]

for key in required_components:
    if key in checkpoint["net"]:
        component_path = os.path.join(temp_dir, f"{key}.pth")
        torch.save(checkpoint["net"][key], component_path)
        print(f"Saved {key} to {component_path}")
    else:
        print(f"Warning: {key} not found in checkpoint")

# Save checkpoint
checkpoint_save_path = os.path.join(temp_dir, "checkpoint.pth")
torch.save(checkpoint, checkpoint_save_path)
print(f"Saved full checkpoint to {checkpoint_save_path}")

# Copy utility models and configs
print("Copying utility models and configs...")

# ASR
shutil.copy("Utils/ASR/epoch_00080.pth", os.path.join(temp_dir, "Utils/ASR/"))
shutil.copy("Utils/ASR/config.yml", os.path.join(temp_dir, "Utils/ASR/"))
shutil.copy("Utils/ASR/models.py", os.path.join(temp_dir, "Utils/ASR/"))
shutil.copy("Utils/ASR/layers.py", os.path.join(temp_dir, "Utils/ASR/"))

# JDC (F0)
shutil.copy("Utils/JDC/bst.t7", os.path.join(temp_dir, "Utils/JDC/"))
shutil.copy("Utils/JDC/model.py", os.path.join(temp_dir, "Utils/JDC/"))

# PLBERT
shutil.copy("Utils/PLBERT/step_1000000.t7", os.path.join(temp_dir, "Utils/PLBERT/"))
shutil.copy("Utils/PLBERT/config.yml", os.path.join(temp_dir, "Utils/PLBERT/"))
shutil.copy("Utils/PLBERT/util.py", os.path.join(temp_dir, "Utils/PLBERT/"))

# Copy necessary Python modules
print("Copying Python modules...")
shutil.copy("text_utils.py", os.path.join(temp_dir, "text_utils.py"))
shutil.copy("models.py", os.path.join(temp_dir, "models.py"))
shutil.copy("utils.py", os.path.join(temp_dir, "utils.py"))

# Plot and save training metrics
log_file = os.path.join(checkpoint_dir, "train.log")
metrics = plot_training_metrics(log_file, temp_dir)

# Save configs
# 1. Save minimal config.yml required by inference
inference_config = {
    "model_params": config["model_params"],
    "preprocess_params": config["preprocess_params"],
    "F0_path": "Utils/JDC/bst.t7",
    "ASR_config": "Utils/ASR/config.yml",
    "ASR_path": "Utils/ASR/epoch_00080.pth",
    "PLBERT_dir": "Utils/PLBERT/",
}
with open(os.path.join(temp_dir, "config.yml"), "w") as f:
    yaml.dump(inference_config, f, default_flow_style=False)

# 2. Save detailed config.json
config_save = {
    "model_params": convert_tensor_to_list(config["model_params"]),
    "training_config": {
        "epochs": config["epochs"],
        "batch_size": config["batch_size"],
        "max_len": config["max_len"],
        "optimizer": config["optimizer_params"],
        "loss_params": config["loss_params"],
    },
    "preprocess_params": convert_tensor_to_list(config["preprocess_params"]),
    "data_params": convert_tensor_to_list(config["data_params"]),
    "model_state": {
        "epoch": int(checkpoint.get("epoch", 0)),
        "iterations": int(checkpoint.get("iters", 0)),
        "val_loss": float(checkpoint.get("val_loss", 0.0)),
    },
    "training_metrics": metrics if metrics else {},
}

with open(os.path.join(temp_dir, "config.json"), "w") as f:
    json.dump(config_save, f, indent=2)

# Create model card with training metrics and inference instructions
model_card = f"""---
language: en
tags:
- text-to-speech
- StyleTTS2
- speech-synthesis
license: mit
pipeline_tag: text-to-speech
---

# StyleTTS2 Fine-tuned Model

This model is a fine-tuned version of StyleTTS2, containing all necessary components for inference.

## Model Details
- **Base Model:** StyleTTS2-LibriTTS
- **Architecture:** StyleTTS2
- **Task:** Text-to-Speech
- **Last Checkpoint:** {latest_checkpoint}

## Training Details
- **Total Epochs:** {config['epochs']}
- **Completed Epochs:** {checkpoint.get('epoch', 0)}
- **Total Iterations:** {checkpoint.get('iters', 0)}
- **Batch Size:** {config['batch_size']}
- **Max Length:** {config['max_len']}
- **Learning Rate:** {config['optimizer_params']['lr']}
- **Final Validation Loss:** {checkpoint.get('val_loss', 0.0):.6f}

## Model Components
The repository includes all necessary components for inference:

### Main Model Components:
"""

for key in checkpoint["net"].keys():
    model_card += f"- {key}.pth\n"

model_card += """
### Utility Components:
- ASR (Automatic Speech Recognition)
  - epoch_00080.pth
  - config.yml
  - models.py
  - layers.py
- JDC (F0 Prediction)
  - bst.t7
  - model.py
- PLBERT
  - step_1000000.t7
  - config.yml
  - util.py

### Additional Files:
- text_utils.py: Text preprocessing utilities
- models.py: Model architecture definitions
- utils.py: Utility functions
- config.yml: Model configuration
- config.json: Detailed configuration and training metrics

## Training Metrics
Training metrics visualization is available in training_metrics.png

## Directory Structure
├── Utils/
│ ├── ASR/
│ ├── JDC/
│ └── PLBERT/
├── model_components/
└── configs/

## Usage Instructions
1. Load the model using the provided config.yml
2. Ensure all utility components (ASR, JDC, PLBERT) are in their respective directories
3. Use text_utils.py for text preprocessing
4. Follow the inference example in the StyleTTS2 documentation
"""

with open(os.path.join(temp_dir, "README.md"), "w") as f:
    f.write(model_card)

print("\nPreparing to upload to Hugging Face...")
print(f"Files to be uploaded from {temp_dir}:")
for root, _, files in os.walk(temp_dir):
    for file in files:
        print(f"- {os.path.relpath(os.path.join(root, file), temp_dir)}")

try:
    # Upload all files in a single commit
    api.upload_folder(
        folder_path=temp_dir,
        repo_id=repo_name,
        repo_type="model",
        commit_message=f"Upload StyleTTS2 checkpoint {latest_checkpoint} with all inference components",
        ignore_patterns=["*.pyc", "__pycache__"],
    )
    print("\nAll files uploaded successfully!")

    # Verify upload
    files = api.list_repo_files(repo_id=repo_name)
    print("\nFiles in repository:")
    for file in files:
        print(f"- {file}")

except Exception as e:
    print(f"Error during upload: {str(e)}")
finally:
    print("\nCleaning up...")
    shutil.rmtree(temp_dir)
    print(f"\nModel uploaded to: https://huggingface.co/{repo_name}")

Loading checkpoint: Models/LJSpeech/epoch_2nd_00004.pth


<ipython-input-13-93fea751455f>:113: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")


Saving model components...
Saved bert to temp_model/bert.pth
Saved bert_encoder to temp_model/bert_encoder.pth
Saved decoder to temp_model/decoder.pth
Saved diffusion to temp_model/diffusion.pth
Saved mpd to temp_model/mpd.pth
Saved msd to temp_model/msd.pth
Saved predictor to temp_model/predictor.pth
Saved predictor_encoder to temp_model/predictor_encoder.pth
Saved style_encoder to temp_model/style_encoder.pth
Saved text_aligner to temp_model/text_aligner.pth
Saved text_encoder to temp_model/text_encoder.pth
Saved pitch_extractor to temp_model/pitch_extractor.pth
Saved wd to temp_model/wd.pth
Saved full checkpoint to temp_model/checkpoint.pth
Copying utility models and configs...
Copying Python modules...

Preparing to upload to Hugging Face...
Files to be uploaded from temp_model:
- bert_encoder.pth
- training_metrics.png
- predictor_encoder.pth
- msd.pth
- utils.py
- mpd.pth
- checkpoint.pth
- text_aligner.pth
- diffusion.pth
- README.md
- wd.pth
- text_encoder.pth
- models.py
- .gi

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


bst.t7:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

...

style_encoder.pth:   0%|          | 0.00/55.5M [00:00<?, ?B/s]

text_aligner.pth:   0%|          | 0.00/31.5M [00:00<?, ?B/s]

text_encoder.pth:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

wd.pth:   0%|          | 0.00/4.70M [00:00<?, ?B/s]


All files uploaded successfully!

Files in repository:
- .gitattributes
- README.md
- Utils/ASR/config.yml
- Utils/ASR/epoch_00080.pth
- Utils/ASR/layers.py
- Utils/ASR/models.py
- Utils/JDC/bst.t7
- Utils/JDC/model.py
- Utils/PLBERT/config.yml
- Utils/PLBERT/step_1000000.t7
- Utils/PLBERT/util.py
- bert.pth
- bert_encoder.pth
- checkpoint.pth
- config.json
- config.yml
- decoder.pth
- diffusion.pth
- models.py
- mpd.pth
- msd.pth
- pitch_extractor.pth
- predictor.pth
- predictor_encoder.pth
- style_encoder.pth
- text_aligner.pth
- text_encoder.pth
- text_utils.py
- training_metrics.png
- utils.py
- wd.pth

Cleaning up...

Model uploaded to: https://huggingface.co/nonoJDWAOIDAWKDA/new2_ft_StyleTTS2
